IMPORT LIBRARIES

In [ ]:
import pandas as pd
import numpy as np

LOADING DATASET


In [ ]:
df = pd.read_csv('/content/FINAL_SQL.csv')

/tmp/ipykernel_2510/49944359.py:1: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/content/FINAL_SQL.csv')


CONVERTING COLUMN NAMES TO STANDARD FORMAT


In [ ]:
df.columns = df.columns.str.strip().str.lower()

CONVERT INVOICE DATE INTO DATETIME FORMAT


In [ ]:
df['invoicedate'] = pd.to_datetime(df['invoicedate'], errors = 'coerce')

CHECKING DATASET


In [ ]:
print(df.shape)
df.info()

(1033036, 9)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1033036 entries, 0 to 1033035
Data columns (total 9 columns):
 #   Column        Non-Null Count    Dtype         
---  ------        --------------    -----         
 0   invoice       1033036 non-null  object        
 1   stockcode     1033036 non-null  object        
 2   description   1028761 non-null  object        
 3   quantity      1033036 non-null  int64         
 4   invoicedate   1033036 non-null  datetime64[ns]
 5   price         1033036 non-null  float64       
 6   customer_id   797885 non-null   float64       
 7   country       1033036 non-null  object        
 8   source_sheet  1033036 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(5)
memory usage: 70.9+ MB


FILTER AND CALCULATE RECENCY, FREQUENCY, MONETARY


In [ ]:
rfm_data = df[(df['customer_id'].notna()) & (df['quantity'] > 0)].copy()
snapshot_date = pd.to_datetime('2011-12-10')
rfm = rfm_data.groupby('customer_id').agg(
    Recency = ('invoicedate', lambda x: (snapshot_date - x.max()).days),
    Frequency = ('invoice', 'nunique'),
    Monetary = ('invoice', lambda x: (
        rfm_data.loc[x.index, 'price']*
        rfm_data.loc[x.index, 'quantity']
    ).sum())).reset_index()


DISPLAY RFM DATA


In [ ]:
display(rfm.head())
print("Total Customer: ", rfm.shape[0])

,customer_id,Recency,Frequency,Monetary
0,12346.0,325,12,77556.46
1,12347.0,2,8,4921.53
2,12348.0,75,5,2019.40
3,12349.0,18,4,4428.69
4,12350.0,310,1,334.40


Total Customer:  5881


SCORE CUSTOMER


In [ ]:
rfm['R_Score'] = pd.qcut(rfm['Recency'], 5, labels = [5, 4, 3, 2, 1]).astype(int)
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 5, labels = [1, 2, 3, 4, 5]).astype(int)
rfm['M_Score'] = pd.qcut(rfm['Monetary'], 5, labels = [1, 2, 3, 4, 5]).astype(int)

ASSIGING BUSINESS SEGMENTS


In [ ]:
def segment_customer(row):
    if row['R_Score'] >= 4 and row['F_Score'] >= 4 and row['M_Score'] >= 4:
        return 'Champions'
    elif row['R_Score'] >= 3 and row['F_Score'] >= 3:
        return 'Loyal Customers'
    elif row['R_Score'] >= 2 and row['F_Score'] >= 3:
        return 'At Risk'
    elif row['R_Score'] >= 4 and row['F_Score'] <= 2:
        return 'New'
    elif row['R_Score'] == 3 and row['F_Score'] <= 2:
        return 'Needs Attention'
    else:
        return 'Lost'
rfm['Segment'] = rfm.apply(segment_customer, axis = 1)

IDENTIFY CHURNED CUSTOMERS


In [ ]:
rfm['IsChurned'] = rfm['Recency'] > 90

### **MERGING RFM RESULTS BACK ONTO TRANSACTIONS**

FILTER VALID CUSTOMER TRANSACTIONS

In [ ]:
df = df[(df['customer_id'].notna()) & (df['quantity'] > 0)].copy()

CREATE INVOICE MONTH

In [ ]:
df['InvoiceMonth'] = (df['invoicedate'].dt.to_period('M').dt.to_timestamp())

CALCULATE REVENUE

In [ ]:
# CALCULATE REVENUE
df['Revenue'] = df['quantity'] * df['price']

df_final = df.merge(rfm[['customer_id', 'Recency', 'Frequency', 'Monetary',
         'R_Score', 'F_Score', 'M_Score', 'Segment', 'IsChurned']],
    on='customer_id', how='left'
)


CHECKING FINAL OUTPUT COLUMNS


In [ ]:
print(df_final.shape)
print(df_final.columns.tolist())
display(df_final.head())

(779495, 19)
['invoice', 'stockcode', 'description', 'quantity', 'invoicedate', 'price', 'customer_id', 'country', 'source_sheet', 'InvoiceMonth', 'Revenue', 'Recency', 'Frequency', 'Monetary', 'R_Score', 'F_Score', 'M_Score', 'Segment', 'IsChurned']


,invoice,stockcode,description,quantity,invoicedate,price,customer_id,country,source_sheet,InvoiceMonth,Revenue,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,Segment,IsChurned
0,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,2010-2011,2009-12-01,30.0,157,8,2433.28,3,4,4,Loyal Customers,True
1,489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01 07:45:00,5.95,13085.0,United Kingdom,2010-2011,2009-12-01,59.5,157,8,2433.28,3,4,4,Loyal Customers,True
2,489434,21871,SAVE THE PLANET MUG,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,2010-2011,2009-12-01,30.0,157,8,2433.28,3,4,4,Loyal Customers,True
3,489434,22041,RECORD FRAME 7 SINGLE SIZE,48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,2010-2011,2009-12-01,100.8,157,8,2433.28,3,4,4,Loyal Customers,True
4,489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01 07:45:00,1.65,13085.0,United Kingdom,2010-2011,2009-12-01,39.6,157,8,2433.28,3,4,4,Loyal Customers,True


CHECKING TOTAL ROWS AND COLUMNS

In [ ]:
print("Total Rows:", df_final.shape[0])
print("Total Columns:", df_final.shape[1])

Total Rows: 779495
Total Columns: 19


EXPORTING FINAL DATASET

In [ ]:
df_final.to_csv('customerAnalyticsRFManalysis_DASHBOARDdata.csv', index = False)

CHECKING CUSTOMER SEGMENT DISTRIBUTION


In [ ]:
print("CUSTOMER COUNT BY SEGMENT")

display(
    rfm['Segment']
    .value_counts()
    .reset_index()
    .rename(columns={
        'index': 'Segment',
        'Segment': 'Customer Count'
    })
)

CUSTOMER COUNT BY SEGMENT


,Customer Count,count
0,Lost,1795
1,Loyal Customers,1416
2,Champions,1291
3,At Risk,550
4,New,442
5,Needs Attention,387


In [ ]:
print("Total RFM Customers:", len(rfm))

print("Customers with missing segments:",
      rfm['Segment'].isna().sum())

print("\nSegment distribution including missing values:")
print(rfm['Segment'].value_counts(dropna=False))

print("\nUnique customer IDs:", rfm['customer_id'].nunique())

Total RFM Customers: 5881
Customers with missing segments: 0

Segment distribution including missing values:
Segment
Lost               1795
Loyal Customers    1416
Champions          1291
At Risk             550
New                 442
Needs Attention     387
Name: count, dtype: int64

Unique customer IDs: 5881


DOWNLOADING CSV

In [ ]:
from google.colab import files

files.download('customerAnalyticsRFManalysis_DASHBOARDdata.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>